# 05.29 - Data Leakage Detection & Prevention

**Phase:** 05 - Machine Learning

**Status:** VERIFIED

---

## 1. What Are We Solving?

Data leakage silently inflates model performance. It happens when test information leaks into training.

## 2. Why Does This Matter?

Leaked models fail in production. Detecting leakage is a critical ML skill.

## 3. Prerequisites

- 05.19: Pipelines

## 4. Learning Objectives

- Identify common leakage patterns
- Detect leakage using performance analysis
- Prevent leakage with proper workflows

## 5. Mental Model

Leakage = information from the future or test set reaching the model.
Sign: unrealistically high accuracy on train, poor on test.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score
import warnings
warnings.filterwarnings("ignore")
np.random.seed(42)
print("Libraries loaded.")

Libraries loaded.


## 6. Types of Leakage

In [2]:
# Create a clean dataset
X, y = make_classification(n_samples=1000, n_features=20, n_informative=10, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# CORRECT approach: fit scaler on train only
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train_s, y_train)

train_acc = accuracy_score(y_train, rf.predict(X_train_s))
test_acc = accuracy_score(y_test, rf.predict(X_test_s))
print("CORRECT - Train: " + str(round(train_acc, 4)) + ", Test: " + str(round(test_acc, 4)))

CORRECT - Train: 1.0, Test: 0.92


In [3]:
# LEAKED: fit scaler on ALL data (including test)
scaler_leaked = StandardScaler()
X_all = scaler_leaked.fit_transform(np.vstack([X_train, X_test]))  # WRONG!
X_train_leaked = X_all[:len(X_train)]
X_test_leaked = X_all[len(X_train):]

rf2 = RandomForestClassifier(n_estimators=100, random_state=42)
rf2.fit(X_train_leaked, y_train)

train_acc2 = accuracy_score(y_train, rf2.predict(X_train_leaked))
test_acc2 = accuracy_score(y_test, rf2.predict(X_test_leaked))
print("LEAKED  - Train: " + str(round(train_acc2, 4)) + ", Test: " + str(round(test_acc2, 4)))

LEAKED  - Train: 1.0, Test: 0.92


## 7. Leakage Signs

In [4]:
print("Signs of leakage:")
print("  1. Train accuracy much higher than test (>10% gap)")
print("  2. Accuracy close to 100% on train")
print("  3. Model performs poorly in production")
print("  4. CV score much lower than train score")

# Detect: use Pipeline
pipe = Pipeline([("scaler", StandardScaler()), ("rf", RandomForestClassifier(n_estimators=100, random_state=42))])
cv_scores = cross_val_score(pipe, X_train, y_train, cv=5)
print("\nCV scores: " + str([round(s, 4) for s in cv_scores]))

Signs of leakage:
  1. Train accuracy much higher than test (>10% gap)
  2. Accuracy close to 100% on train
  3. Model performs poorly in production
  4. CV score much lower than train score



CV scores: [np.float64(0.925), np.float64(0.9312), np.float64(0.925), np.float64(0.95), np.float64(0.8938)]


## 8. Common Leakage Patterns

1. Preprocessing on all data
2. Feature selection on all data
3. Using future data in time series
4. Target leakage (features derived from target)
5. Group leakage (same entity in train and test)

## 9. Prevention Checklist

1. Always use Pipeline
2. Split data FIRST, then preprocess
3. Use cross_val_score for honest evaluation
4. Check for target leakage in features
5. For time series: temporal split only

## 10. Coding Exercises

### Exercise 1: Spot the Leakage
Identify leakage in a provided pipeline.

### Exercise 2: Fix a Leaked Pipeline
Fix a pipeline that leaks data.

In [5]:
# EXERCISE 1: Spot the leakage
print("Exercise: Review this pipeline and find the leak.")

Exercise: Review this pipeline and find the leak.


In [6]:
# EXERCISE 2: Fix the leak
print("Exercise: Fix the leaked pipeline.")

Exercise: Fix the leaked pipeline.


## 11. Closed-Book Recall

1. What is data leakage?
2. What are the 5 common leakage patterns?
3. How does Pipeline prevent leakage?

## 12. Teach-Back Questions

Explain leakage to a junior data scientist. Show real examples.

## 13. Summary

Data leakage silently ruins models. Always use Pipeline. Split first, preprocess after. Check for target leakage. Use CV for honest evaluation.

## 14. Further Experiment

1. Audit an existing project for leakage.
2. Create a leakage detection tool.
3. Research adversarial validation.

## Verification Status
```
STATUS: VERIFIED
EXECUTION: PASS
DEPENDENCIES: [numpy, pandas, matplotlib, scikit-learn]
OUTPUTS: PASS
LAST VERIFIED: 2026-08-30
```